In [ ]:
# src/eda.py

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

# Define directories for data and output plots
DATA_DIR = Path("../data")
PLOT_DIR = Path("../plot")
PLOT_DIR.mkdir(exist_ok=True)

# Load feature names and assign unique identifiers
features = pd.read_csv(DATA_DIR / "features.txt", sep="\s+", header=None, names=['index', 'feature'])
feature_names_unique = [f"{f}_{i}" for i, f in enumerate(features['feature'].values)]

# Load activity label mappings
activity_labels = pd.read_csv(DATA_DIR / "activity_labels.txt", sep="\s+", header=None, names=['id', 'activity'])

# Load training data and map activity IDs to descriptive labels
X_train = pd.read_csv(DATA_DIR / "train/X_train.txt", sep="\s+", header=None, names=feature_names_unique)
y_train = pd.read_csv(DATA_DIR / "train/y_train.txt", sep="\s+", header=None, names=['Activity'])
y_train['Activity'] = y_train['Activity'].map(activity_labels.set_index('id')['activity'])

# Display dataset overview
print("Training Data Shape:", X_train.shape)
print("\nActivity Distribution:\n", y_train['Activity'].value_counts())

# Select mean() and std() features for EDA
selected_features = [f for f in feature_names_unique if 'mean()' in f or 'std()' in f]

# Activity distribution visualisation
plt.figure(figsize=(8, 4))
sns.countplot(x='Activity', data=y_train, order=activity_labels['activity'])
plt.title("Activity Distribution in Training Set")
plt.ylabel("Number of Samples")
plt.xlabel("Activity")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(PLOT_DIR / "activity_distribution.png")
plt.show()

# Boxplots for selected features
for f in selected_features:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=y_train['Activity'], y=X_train[f])
    plt.title(f"Distribution of {f} Across Activities")
    plt.xlabel("Activity")
    plt.ylabel(f)
    plt.xticks(rotation=45)
    plt.tight_layout()
    safe_name = f.replace("()", "").replace("-", "_").replace(",", "_")
    plt.savefig(PLOT_DIR / f"{safe_name}_boxplot.png")
    plt.close()

# Descriptive statistics for selected features
desc_stats = X_train[selected_features].describe()
desc_stats.to_csv(PLOT_DIR / "descriptive_statistics.csv")

# Correlation heatmap for a subset of features
subset_features = selected_features[:25]
corr_matrix = X_train[subset_features].corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation Heatmap (Subset)")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(PLOT_DIR / "correlation_heatmap_subset.png")
plt.show()

print("EDA complete. Plots and statistics saved in 'plot/' folder.")
